# Homework 8: Neural Networks and Deep Learning

Classify curly vs straight hair with a small PyTorch CNN.
Reference setup: Python 3.11, CPU wheels (`torch==2.9.0+cpu`, `torchvision==0.24.0+cpu`), seed 42.

## Setup — environment

Install the pinned CPU wheels.

In [6]:
!python -m pip install -q torch==2.9.0+cpu torchvision==0.24.0+cpu numpy==2.3.3 Pillow==11.3.0 --index-url https://download.pytorch.org/whl/cpu --extra-index-url https://pypi.org/simple


[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


## Setup — download and verify the dataset (bash)

The archive checksum must match the manifest. We verify it before extracting.

### Download + verify checksum

In [14]:
%%bash
set -e

wget -q -nc https://github.com/SVizor42/ML_Zoomcamp/releases/download/straight-curly-data/data.zip

echo "Expected: 9e53453e3017502f22860cb08558f0cdb343e102fb61feab75960616185e7d67"
echo -n "Actual:   "
sha256sum data.zip | awk '{print $1}'

ACTUAL=$(sha256sum data.zip | awk '{print $1}')
if [ "$ACTUAL" != "9e53453e3017502f22860cb08558f0cdb343e102fb61feab75960616185e7d67" ]; then
  echo "CHECKSUM MISMATCH" >&2
  exit 1
fi
echo "CHECKSUM OK"

Couldn't find program: 'bash'


### Extract and inspect

In [13]:
%%bash
rm -rf data
unzip -q data.zip -d .
echo "-- top-level of data/ --"
ls data
echo "-- counts (should be 800 train, 201 test) --"
find data/train -type f | wc -l
find data/test  -type f | wc -l
echo "-- class folders --"
ls data/train
ls data/test

Couldn't find program: 'bash'


## Reproducibility — seed everything BEFORE building datasets / model / optimizer

As required by the assignment. `num_workers=0` and a dedicated `torch.Generator` for the training loader.

### Imports + seeding + constants

In [11]:
import json, random, statistics
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.use_deterministic_algorithms(True)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cpu")
torch.set_num_threads(2)

BATCH_SIZE = 20
EPOCHS_PER_PHASE = 10

NORMALIZE = transforms.Normalize(
    mean=[0.485, 0.456, 0.406],
    std=[0.229, 0.224, 0.225],
)

DATA_DIR = Path("./data")
OUT_DIR = Path("runs/reference")
print("torch:", torch.__version__, "| device:", device)

torch: 2.9.0+cpu | device: cpu


## Transforms and dataset

Baseline train transform = evaluation transform (no augmentation in phase 1). Evaluation transform is used for the held-out split in both phases.

### Define transforms + datasets + sanity checks

In [15]:
def evaluation_transform():
    return transforms.Compose([
        transforms.Resize(
            (200, 200),
            interpolation=transforms.InterpolationMode.BILINEAR,
        ),
        transforms.ToTensor(),
        NORMALIZE,
    ])


def augmented_train_transform():
    return transforms.Compose([
        transforms.Resize(
            (200, 200),
            interpolation=transforms.InterpolationMode.BILINEAR,
        ),
        transforms.RandomRotation(
            50,
            interpolation=transforms.InterpolationMode.NEAREST,
        ),
        transforms.RandomResizedCrop(
            200,
            scale=(0.9, 1.0),
            ratio=(0.9, 1.1),
            interpolation=transforms.InterpolationMode.BILINEAR,
        ),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        NORMALIZE,
    ])


train_dir = DATA_DIR / "train"
evaluation_dir = DATA_DIR / "test"

baseline_dataset = datasets.ImageFolder(train_dir, transform=evaluation_transform())
evaluation_dataset = datasets.ImageFolder(evaluation_dir, transform=evaluation_transform())

assert len(baseline_dataset) == 800, len(baseline_dataset)
assert len(evaluation_dataset) == 201, len(evaluation_dataset)
assert baseline_dataset.class_to_idx == {"curly": 0, "straight": 1}, baseline_dataset.class_to_idx
print("train:", len(baseline_dataset), "| eval:", len(evaluation_dataset), "| classes:", baseline_dataset.class_to_idx)

train: 800 | eval: 201 | classes: {'curly': 0, 'straight': 1}


## DataLoaders

`num_workers=0` for determinism. Training loader gets its own fixed generator.

### Loader factory + evaluation loader

In [16]:
def make_loader(dataset, *, shuffle: bool, seed: int | None = None):
    options = {
        "batch_size": BATCH_SIZE,
        "shuffle": shuffle,
        "num_workers": 0,
    }
    if seed is not None:
        options["generator"] = torch.Generator().manual_seed(seed)
    return DataLoader(dataset, **options)


evaluation_loader = make_loader(evaluation_dataset, shuffle=False)
print("eval batches:", len(evaluation_loader))

eval batches: 11


## Model — `HairModel`

Emits a single logit. Use `BCEWithLogitsLoss` (sigmoid + BCE fused). Do **not** add a sigmoid in `forward`.

### Model definition + parameter count (Q2)

In [17]:
class HairModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Conv2d(3, 32, kernel_size=3, stride=1, padding=0)
        self.pool = nn.MaxPool2d(kernel_size=2)
        self.hidden = nn.Linear(32 * 99 * 99, 64)
        self.output = nn.Linear(64, 1)

    def forward(self, x):
        x = self.pool(nn.functional.relu(self.conv(x)))
        x = x.flatten(start_dim=1)
        x = nn.functional.relu(self.hidden(x))
        return self.output(x)


model = HairModel().to(device)
n_params = sum(p.numel() for p in model.parameters())
print("trainable parameters:", n_params)
assert n_params == 20_073_473, n_params

trainable parameters: 20073473


## Question 1 — loss and output contract

**Answer: `nn.BCEWithLogitsLoss()`**

The model emits one logit and the loss applies the stable sigmoid + binary cross-entropy operation in one step. `CrossEntropyLoss` expects class logits (no sigmoid), `MSELoss` is for regression, `CosineEmbeddingLoss` is for embedding similarity.

## Question 2 — parameter count

Counted above: **`20073473`**

Breakdown:
- `conv`: 32×3×3×3 weights + 32 bias = 896
- `hidden`: 64 × (32·99·99) weights + 64 bias = 20,072,512
- `output`: 64 weights + 1 bias = 65
- total = 20,073,473

## Training helpers

Evaluation uses `model.eval()` + `torch.no_grad()`. Loss and accuracy are weighted by number of examples, not batches.

In [18]:
def evaluate(model, loader, criterion):
    model.eval()
    loss_sum = 0.0
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            labels = labels.float().unsqueeze(1)
            logits = model(images)
            loss = criterion(logits, labels)
            loss_sum += loss.item() * images.size(0)
            correct += ((torch.sigmoid(logits) >= 0.5) == labels).sum().item()
            total += labels.size(0)
    return loss_sum / total, correct / total


def train_phase(model, loader, evaluation_loader, criterion, optimizer):
    history = {
        "train_loss": [],
        "train_accuracy": [],
        "evaluation_loss": [],
        "evaluation_accuracy": [],
    }
    for epoch in range(EPOCHS_PER_PHASE):
        model.train()
        loss_sum = 0.0
        correct = 0
        total = 0
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            labels = labels.float().unsqueeze(1)
            optimizer.zero_grad()
            logits = model(images)
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()
            loss_sum += loss.item() * images.size(0)
            correct += ((torch.sigmoid(logits) >= 0.5) == labels).sum().item()
            total += labels.size(0)

        evaluation_loss, evaluation_accuracy = evaluate(model, evaluation_loader, criterion)
        history["train_loss"].append(loss_sum / total)
        history["train_accuracy"].append(correct / total)
        history["evaluation_loss"].append(evaluation_loss)
        history["evaluation_accuracy"].append(evaluation_accuracy)

        print(
            f"epoch {epoch + 1:2d} | "
            f"train loss {history['train_loss'][-1]:.4f} | "
            f"train acc {history['train_accuracy'][-1]:.4f} | "
            f"eval loss {evaluation_loss:.4f} | "
            f"eval acc {evaluation_accuracy:.4f}"
        )
    return history

## Phase 1 — baseline (10 epochs)

Criterion and optimizer are created **once** and reused for both phases.

In [19]:
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.002, momentum=0.8)

baseline_history = train_phase(
    model,
    make_loader(baseline_dataset, shuffle=True, seed=SEED),
    evaluation_loader,
    criterion,
    optimizer,
)

epoch  1 | train loss 0.6385 | train acc 0.6450 | eval loss 0.6010 | eval acc 0.6567
epoch  2 | train loss 0.5441 | train acc 0.7175 | eval loss 0.6003 | eval acc 0.6517
epoch  3 | train loss 0.4955 | train acc 0.7550 | eval loss 0.6405 | eval acc 0.6219
epoch  4 | train loss 0.4330 | train acc 0.7913 | eval loss 0.6274 | eval acc 0.6915
epoch  5 | train loss 0.4048 | train acc 0.8087 | eval loss 0.6293 | eval acc 0.6617
epoch  6 | train loss 0.3953 | train acc 0.8063 | eval loss 0.6332 | eval acc 0.7114
epoch  7 | train loss 0.3437 | train acc 0.8363 | eval loss 0.6895 | eval acc 0.6716
epoch  8 | train loss 0.2864 | train acc 0.8750 | eval loss 0.6526 | eval acc 0.7065
epoch  9 | train loss 0.2224 | train acc 0.9125 | eval loss 0.6980 | eval acc 0.7363
epoch 10 | train loss 0.1818 | train acc 0.9213 | eval loss 1.0453 | eval acc 0.6119


### Save baseline history

In [20]:
OUT_DIR.mkdir(parents=True, exist_ok=True)
(OUT_DIR / "history_baseline.json").write_text(
    json.dumps(baseline_history, indent=2) + "\n",
    encoding="utf-8",
)
print((OUT_DIR / "history_baseline.json").read_text())

{
  "train_loss": [
    0.6385456457734108,
    0.5441446743905545,
    0.4955223076045513,
    0.43304194808006286,
    0.40478640906512736,
    0.3953401420265436,
    0.3436661325395107,
    0.2863818446174264,
    0.22242047041654586,
    0.18179294606670737
  ],
  "train_accuracy": [
    0.645,
    0.7175,
    0.755,
    0.79125,
    0.80875,
    0.80625,
    0.83625,
    0.875,
    0.9125,
    0.92125
  ],
  "evaluation_loss": [
    0.6010399336245522,
    0.6002970326006116,
    0.6405199976702828,
    0.627439008423345,
    0.6292903159388262,
    0.6331971283279249,
    0.6895431630053923,
    0.6526027922310046,
    0.6979533510088031,
    1.0453158441203558
  ],
  "evaluation_accuracy": [
    0.6567164179104478,
    0.6517412935323383,
    0.6218905472636815,
    0.6915422885572139,
    0.6616915422885572,
    0.7114427860696517,
    0.6716417910447762,
    0.7064676616915423,
    0.736318407960199,
    0.6119402985074627
  ]
}



## Question 3 — median of `train_accuracy` (baseline)

Submit rounded to 2 decimals; tolerance ±0.05.

### Q3 answer

In [21]:
q3 = statistics.median(baseline_history["train_accuracy"])
print("median train_accuracy:", q3, "-> rounded:", round(q3, 2))

median train_accuracy: 0.8075 -> rounded: 0.81


## Question 4 — population std of `train_loss` (baseline)

Use `np.std(..., ddof=0)`. Round to 3 decimals; tolerance ±0.02.

### Q4 answer

In [22]:
q4 = float(np.std(baseline_history["train_loss"], ddof=0))
print("population std of train_loss:", q4, "-> rounded:", round(q4, 3))

population std of train_loss: 0.1352325929115262 -> rounded: 0.135


## Phase 2 — augmented (10 more epochs)

Reuse the **same model and optimizer**. Only the training transform changes. Evaluation dataset and loader are unchanged and deterministic.

### Augmented dataset + run phase 2

In [23]:
augmented_dataset = datasets.ImageFolder(
    train_dir,
    transform=augmented_train_transform(),
)
assert len(augmented_dataset) == 800

augmented_history = train_phase(
    model,
    make_loader(augmented_dataset, shuffle=True, seed=SEED + 1),
    evaluation_loader,
    criterion,
    optimizer,
)

epoch  1 | train loss 0.6529 | train acc 0.6188 | eval loss 0.6647 | eval acc 0.7015
epoch  2 | train loss 0.5876 | train acc 0.6850 | eval loss 0.6192 | eval acc 0.6866
epoch  3 | train loss 0.5862 | train acc 0.6675 | eval loss 0.6028 | eval acc 0.6716
epoch  4 | train loss 0.5814 | train acc 0.6937 | eval loss 0.5785 | eval acc 0.6766
epoch  5 | train loss 0.5707 | train acc 0.7075 | eval loss 0.5815 | eval acc 0.7114
epoch  6 | train loss 0.5529 | train acc 0.7225 | eval loss 0.5517 | eval acc 0.7164
epoch  7 | train loss 0.5407 | train acc 0.7113 | eval loss 0.5489 | eval acc 0.7363
epoch  8 | train loss 0.5323 | train acc 0.7100 | eval loss 0.5333 | eval acc 0.7363
epoch  9 | train loss 0.4948 | train acc 0.7775 | eval loss 0.6014 | eval acc 0.7114
epoch 10 | train loss 0.5246 | train acc 0.7350 | eval loss 0.5212 | eval acc 0.7612


### Save augmented history

In [24]:
(OUT_DIR / "history_augmented.json").write_text(
    json.dumps(augmented_history, indent=2) + "\n",
    encoding="utf-8",
)
print((OUT_DIR / "history_augmented.json").read_text())

{
  "train_loss": [
    0.6528983145952225,
    0.587592301517725,
    0.586178831756115,
    0.5813827596604824,
    0.5706800602376461,
    0.5529032595455646,
    0.5407039225101471,
    0.5323435485363006,
    0.494847846031189,
    0.5246367886662483
  ],
  "train_accuracy": [
    0.61875,
    0.685,
    0.6675,
    0.69375,
    0.7075,
    0.7225,
    0.71125,
    0.71,
    0.7775,
    0.735
  ],
  "evaluation_loss": [
    0.6647181805983111,
    0.6192319375365528,
    0.602773544133006,
    0.578464333097733,
    0.5815482157379833,
    0.5516569899089301,
    0.5488608328264151,
    0.5332511506270413,
    0.6013608797272639,
    0.5212399805363138
  ],
  "evaluation_accuracy": [
    0.7014925373134329,
    0.6865671641791045,
    0.6716417910447762,
    0.6766169154228856,
    0.7114427860696517,
    0.7164179104477612,
    0.736318407960199,
    0.736318407960199,
    0.7114427860696517,
    0.7611940298507462
  ]
}



## Question 5 — mean of `evaluation_loss` (augmented)

Round to 3 decimals; tolerance ±0.02.

### Q5 answer

In [25]:
q5 = statistics.mean(augmented_history["evaluation_loss"])
print("mean evaluation_loss:", q5, "-> rounded:", round(q5, 3))

mean evaluation_loss: 0.580310604472955 -> rounded: 0.58


## Question 6 — mean of last 5 `evaluation_accuracy` (augmented)

Epochs 6–10 of the augmented phase. Round to 2 decimals; tolerance ±0.05.

### Q6 answer

In [26]:
q6 = statistics.mean(augmented_history["evaluation_accuracy"][-5:])
print("mean last-5 evaluation_accuracy:", q6, "-> rounded:", round(q6, 2))

mean last-5 evaluation_accuracy: 0.7323383084577114 -> rounded: 0.73


## Submit the results

- URL: https://courses.datatalks.club/ml-zoomcamp-2026/homework/hw08
- Include `history_baseline.json` and `history_augmented.json` (in `runs/reference/`) or a link to this notebook.
- Numeric answers must use the precision and tolerance stated in each question.